# Lesson 4b: Gaussian Mixture Models (GMM) — Practical

## Overview
This lesson applies scikit-learn's GaussianMixture to real data, focusing on model selection, hyperparameter tuning, and interpreting soft cluster assignments. We'll segment images and compare GMM to K-Means.

**Learning Goals:**
- Use scikit-learn GaussianMixture API
- Select optimal K via BIC and AIC
- Interpret soft assignments (responsibilities)
- Apply GMM to image segmentation
- Compare covariance types and their impact

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.io import loadmat
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Part 1: Synthetic Dataset with Known Ground Truth

Generate a 3-component mixture for model selection demonstration.

In [ ]:
# Generate 3-component GMM data
np.random.seed(42)
n_samples = 500

# Component 1: centered at (-3, -3), spherical
c1 = np.random.multivariate_normal([-3, -3], [[0.8, 0], [0, 0.8]], n_samples // 3)

# Component 2: centered at (0, 3), elongated
c2 = np.random.multivariate_normal([0, 3], [[2, 0.8], [0.8, 0.5]], n_samples // 3)

# Component 3: centered at (3, 0), spherical
c3 = np.random.multivariate_normal([3, 0], [[0.6, 0], [0, 0.6]], n_samples - 2 * (n_samples // 3))

X = np.vstack([c1, c2, c3])
y_true = np.hstack([np.zeros(n_samples // 3), np.ones(n_samples // 3), 2 * np.ones(n_samples - 2 * (n_samples // 3))])

# Scatter plot
plt.figure(figsize=(10, 4))
for k in range(3):
    plt.scatter(X[y_true == k, 0], X[y_true == k, 1], label=f'Component {k+1}', alpha=0.6)
plt.title('3-Component Synthetic Data')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.legend()
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"Data shape: {X.shape}")
print(f"Component sizes: {np.sum(y_true == 0)}, {np.sum(y_true == 1)}, {np.sum(y_true == 2)}")

## Part 2: Model Selection with BIC and AIC

### Information Criteria
**BIC (Bayesian Information Criterion):**
$$BIC = -2 \log L + k \log n$$

**AIC (Akaike Information Criterion):**
$$AIC = -2 \log L + 2k$$

where $L$ is the maximum likelihood, $k$ is the number of parameters, $n$ is the number of observations.

**Lower is better** (trade-off between fit quality and model complexity).

In [ ]:
# Try different numbers of components
n_components_range = range(1, 10)
bic_scores = []
aic_scores = []
silhouette_scores = []

for n_components in n_components_range:
    # Fit GMM
    gmm = GaussianMixture(n_components=n_components, n_init=10, random_state=42)
    gmm.fit(X)
    
    # Compute criteria
    bic_scores.append(gmm.bic(X))
    aic_scores.append(gmm.aic(X))
    
    # Hard assignments for silhouette
    labels = gmm.predict(X)
    if n_components > 1:
        silhouette_scores.append(silhouette_score(X, labels))
    else:
        silhouette_scores.append(0)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# BIC
axes[0].plot(n_components_range, bic_scores, 'b-o', linewidth=2, markersize=6)
axes[0].axvline(x=3, color='r', linestyle='--', label='True K=3')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('BIC')
axes[0].set_title('BIC (lower is better)')
axes[0].grid(True)
axes[0].legend()

# AIC
axes[1].plot(n_components_range, aic_scores, 'g-o', linewidth=2, markersize=6)
axes[1].axvline(x=3, color='r', linestyle='--', label='True K=3')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('AIC')
axes[1].set_title('AIC (lower is better)')
axes[1].grid(True)
axes[1].legend()

# Silhouette
axes[2].plot(n_components_range, silhouette_scores, 'm-o', linewidth=2, markersize=6)
axes[2].axvline(x=3, color='r', linestyle='--', label='True K=3')
axes[2].set_xlabel('Number of Components')
axes[2].set_ylabel('Silhouette Score')
axes[2].set_title('Silhouette Score (higher is better)')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

# Find optimal K
bic_optimal = n_components_range[np.argmin(bic_scores)]
aic_optimal = n_components_range[np.argmin(aic_scores)]
silhouette_optimal = n_components_range[np.argmax(silhouette_scores)]

print(f"BIC recommends K={bic_optimal}")
print(f"AIC recommends K={aic_optimal}")
print(f"Silhouette recommends K={silhouette_optimal}")
print(f"True K=3")

## Part 3: Soft Assignments (Responsibilities)

Unlike K-Means, GMM provides soft assignments: probabilities of cluster membership.

In [ ]:
# Fit optimal GMM
gmm_optimal = GaussianMixture(n_components=3, n_init=10, random_state=42)
gmm_optimal.fit(X)

# Get soft assignments (responsibilities)
responsibilities = gmm_optimal.predict_proba(X)

# Get hard assignments
labels_hard = gmm_optimal.predict(X)

print(f"Responsibilities shape: {responsibilities.shape}")
print(f"\nFirst 5 samples' responsibilities (probabilities for each component):")
print(responsibilities[:5])

# Identify "uncertain" points (max responsibility < 0.9)
max_responsibility = responsibilities.max(axis=1)
uncertain = max_responsibility < 0.9
print(f"\nNumber of uncertain assignments (max resp < 0.9): {np.sum(uncertain)} / {len(X)}")

# Visualize soft assignments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hard assignments
ax = axes[0]
scatter = ax.scatter(X[:, 0], X[:, 1], c=labels_hard, cmap='viridis', s=30, alpha=0.6)
ax.set_title('Hard Assignments (argmax of responsibilities)')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
plt.colorbar(scatter, ax=ax, label='Cluster')

# Soft assignments: color by max responsibility (uncertainty)
ax = axes[1]
scatter = ax.scatter(X[:, 0], X[:, 1], c=max_responsibility, cmap='RdYlGn', s=30, alpha=0.6)
ax.set_title('Maximum Responsibility (confidence in assignment)')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
cbar = plt.colorbar(scatter, ax=ax, label='Max Responsibility')
cbar.set_label('Max Responsibility (closer to 1 = more confident)')

plt.tight_layout()
plt.show()

## Part 4: Covariance Types Comparison

Compare 'full', 'tied', 'diag', and 'spherical' covariance structures.

In [ ]:
covariance_types = ['full', 'tied', 'diag', 'spherical']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, cov_type in enumerate(covariance_types):
    gmm = GaussianMixture(n_components=3, covariance_type=cov_type, n_init=10, random_state=42)
    gmm.fit(X)
    labels = gmm.predict(X)
    
    ax = axes[i]
    for k in range(3):
        ax.scatter(X[labels == k, 0], X[labels == k, 1], label=f'Cluster {k+1}', alpha=0.6)
    
    # Plot ellipses representing Gaussians
    from matplotlib.patches import Ellipse
    for k in range(3):
        mean = gmm.means_[k]
        cov = gmm.covariances_[k] if gmm.covariances_.ndim == 3 else np.diag(gmm.covariances_[k])
        
        # Eigendecomposition for ellipse
        eigvals, eigvecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
        width, height = 4 * np.sqrt(eigvals)
        ellipse = Ellipse(mean, width, height, angle=angle, fill=False, edgecolor='k', linewidth=1.5)
        ax.add_patch(ellipse)
    
    bic = gmm.bic(X)
    ax.set_title(f'{cov_type.capitalize()} (BIC={bic:.0f})')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.axis('equal')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 5)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Part 5: Image Segmentation with GMM

Apply GMM to image pixel colors for segmentation.

In [ ]:
# Create a simple synthetic image with 3 color regions
np.random.seed(42)
height, width = 100, 150
image = np.zeros((height, width, 3))

# Red region (top-left)
image[:50, :75] = [1, 0, 0]
image[:50, :75] += np.random.normal(0, 0.05, (*image[:50, :75].shape,))

# Green region (top-right)
image[:50, 75:] = [0, 1, 0]
image[:50, 75:] += np.random.normal(0, 0.05, (*image[:50, 75:].shape,))

# Blue region (bottom)
image[50:] = [0, 0, 1]
image[50:] += np.random.normal(0, 0.05, (*image[50:].shape,))

# Clip to valid range
image = np.clip(image, 0, 1)

# Reshape for clustering
X_image = image.reshape(-1, 3)

# Fit GMM
gmm_image = GaussianMixture(n_components=3, n_init=10, random_state=42)
labels_image = gmm_image.fit_predict(X_image)
responsibilities_image = gmm_image.predict_proba(X_image)

# Reshape back to image
segmented = labels_image.reshape(height, width)
max_responsibility_image = responsibilities_image.max(axis=1).reshape(height, width)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(image)
axes[0].set_title('Original Image (RGB)')
axes[0].axis('off')

axes[1].imshow(segmented, cmap='viridis')
axes[1].set_title('GMM Segmentation (Hard Assignment)')
axes[1].axis('off')

im = axes[2].imshow(max_responsibility_image, cmap='RdYlGn')
axes[2].set_title('Segmentation Confidence')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], label='Max Responsibility')

plt.tight_layout()
plt.show()

print(f"Image shape: {image.shape}")
print(f"Segmented clusters: {np.unique(segmented)}")
print(f"Average confidence: {max_responsibility_image.mean():.3f}")

## Part 6: GMM vs K-Means on Real Data

Compare both methods' performance and interpretability.

In [ ]:
# Fit both models
gmm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gmm.fit(X)
gmm_labels = gmm.predict(X)
gmm_ll = gmm.score(X) * len(X)  # Total log-likelihood

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans.fit(X)
kmeans_labels = kmeans.labels_

# Metrics
gmm_silhouette = silhouette_score(X, gmm_labels)
kmeans_silhouette = silhouette_score(X, kmeans_labels)
gmm_davies_bouldin = davies_bouldin_score(X, gmm_labels)
kmeans_davies_bouldin = davies_bouldin_score(X, kmeans_labels)

# Comparison table
print("\n" + "="*50)
print("GMM vs K-Means Comparison")
print("="*50)
print(f"{'Metric':<25} {'GMM':<15} {'K-Means':<15}")
print("-"*50)
print(f"{'Log-Likelihood':<25} {gmm_ll:<15.2f} {'N/A':<15}")
print(f"{'BIC':<25} {gmm.bic(X):<15.2f} {'N/A':<15}")
print(f"{'Silhouette Score':<25} {gmm_silhouette:<15.3f} {kmeans_silhouette:<15.3f}")
print(f"{'Davies-Bouldin Index':<25} {gmm_davies_bouldin:<15.3f} {kmeans_davies_bouldin:<15.3f}")
print("="*50)
print("\nNote: Silhouette higher is better; Davies-Bouldin lower is better")

## Part 7: Extracting and Interpreting Parameters

In [ ]:
# Extract GMM parameters
weights = gmm.weights_
means = gmm.means_
covariances = gmm.covariances_

print("Learned GMM Parameters:")
print("\nMixing Weights (π):")
for k, w in enumerate(weights):
    print(f"  Component {k+1}: {w:.3f}")

print("\nMeans (μ):")
for k, m in enumerate(means):
    print(f"  Component {k+1}: {m}")

print("\nCovariance Matrices (Σ):")
for k, cov in enumerate(covariances):
    print(f"  Component {k+1}:")
    print(f"    {cov}")

# Check convergence
print(f"\nConverged: {gmm.converged_}")
print(f"Number of iterations: {gmm.n_iter_}")
print(f"Lower bound on log-likelihood: {gmm.lower_bound_:.2f}")

## Summary

### Key Takeaways
1. **Model Selection**: BIC/AIC automatically penalize complexity; effective for choosing K
2. **Soft Assignments**: Responsibilities give uncertainty quantification, unlike K-Means
3. **Covariance Types**: Full covariance is most flexible; spherical is most constrained
4. **Image Segmentation**: GMM naturally handles mixed colors and boundary uncertainty
5. **Interpretability**: Extracted parameters (weights, means, covariances) are directly interpretable

### Practical Workflow
1. Fit GMM with different K values
2. Use BIC to select optimal K
3. Extract soft assignments for uncertainty-aware applications
4. Visualize learned cluster shapes via covariance matrices
5. Compare with K-Means for intuition about soft vs hard assignments

In [ ]:
# Exercise 1: Grid search over covariance types and K
print("\nExercise: Grid search over covariance types and K")
best_bic = np.inf
best_config = None

for cov_type in ['full', 'diag', 'spherical']:
    for K in range(2, 6):
        gmm = GaussianMixture(n_components=K, covariance_type=cov_type, n_init=5, random_state=42)
        gmm.fit(X)
        bic = gmm.bic(X)
        if bic < best_bic:
            best_bic = bic
            best_config = (cov_type, K)
        print(f"  {cov_type:10} K={K}: BIC={bic:.0f}")

print(f"\nBest configuration: {best_config[0]} with K={best_config[1]} (BIC={best_bic:.0f})")

# Exercise 2: Analyze boundary points
print("\n\nExercise: Points on cluster boundaries")
max_resp = responsibilities_image.max(axis=1)
boundary_points = np.where(max_resp < 0.7)[0]
interior_points = np.where(max_resp >= 0.7)[0]
print(f"Boundary points (confidence < 0.7): {len(boundary_points)} / {len(max_resp)}")
print(f"Interior points (confidence >= 0.7): {len(interior_points)} / {len(max_resp)}")